In [1]:
# ==============================================================================
# Практична робота 20.
# Варіант 6: Чернігів

"""
ЗАВДАННЯ 1. Математична постановка задачі (Чернігів)
--------------------------------------------------------------------------------
Змінні рішення:
    x1 — кількість вироблених одиниць виробу А за тиждень (од/тижд)
    x2 — кількість вироблених одиниць виробу Б за тиждень (од/тижд)

Цільова функція (максимізація тижневого прибутку, грн):
    max Z = 55*x1 + 40*x2

Ресурсні обмеження (нерівності <=):
    1) Сировина (кг):              2*x1 + 3*x2 <= 210
    2) Час обладнання (маш-год):    6*x1 + 2*x2 <= 250
    3) Електроенергія (кВт·год):   2*x1 + 1*x2 <= 125

Умова невід'ємності:
    x1 >= 0,  x2 >= 0
--------------------------------------------------------------------------------
"""

from scipy.optimize import linprog

# ==============================================================================
# ЗАВДАННЯ 2. Розв'язання через scipy.optimize.linprog
# ==============================================================================

# 1. Коефіцієнти цільової функції.
# Оскільки linprog мінімізує, для максимізації інвертуємо знаки: min (-Z)
c = [-55, -40]

# 2. Матриця коефіцієнтів лівих частин обмежень (A_ub @ x <= b_ub)
A_ub = [
    [2, 3],  # Сировина
    [6, 2],  # Час обладнання
    [2, 1],  # Електроенергія
]

# 3. Вектор правих частин обмежень (ліміти ресурсів)
b_ub = [210, 250, 125]

# 4. Границі для змінних (x1 >= 0, x2 >= 0)
bounds = [(0, None), (0, None)]

# 5. Виклик оптимізатора HiGHS
res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")

# Виведення первинних результатів
print("=== ЗАВДАННЯ 2. Результати linprog ===")
print(f"res.x (одиниці А та Б): {res.x}")
print(f"res.fun (значення мінус-функції): {res.fun}")
print(f"Дійсний максимальний прибуток (-res.fun): {-res.fun:.2f} грн")
print(f"res.status: {res.status}")
print(f"res.message: {res.message}\n")

# Підтвердження статусу розв'язку
if res.status == 0:
    print("ПІДТВЕРДЖЕННЯ: res.status == 0 (Глобальний оптимум успішно знайдено).\n")

"""
ЗАВДАННЯ 3. Інтерпретація оптимального розв'язку
--------------------------------------------------------------------------------
Результати:
    - Виріб А (x1): 23.75 одиниць на тиждень
    - Виріб Б (x2): 54.167 одиниць на тиждень (54 і 1/6)
    - Максимальний прибуток: 3472.92 грн на тиждень

Пояснення щодо цілочисловості:
    Отримані значення є дробовими, оскільки класичне лінійне програмування працює 
    у неперервній області дійсних чисел. У реальному виробництві випускають цілі 
    одиниці продукції.
    Просте округлення (наприклад, x1=24, x2=54) НЕ є надійним підходом:
    1) Округлення "вгору" може вивести точку поза межі ресурсних лімітів (розв'язок
       стане недопустимим).
    2) Округлення "вниз" збереже припустимість, але не гарантує досягнення 
       максимально можливого прибутку серед усіх можливих цілих точок.
    Для точного цілочислового розв'язку слід використовувати алгоритми змішаного
    цілочислового лінійного програмування (MILP, наприклад scipy.optimize.milp).
--------------------------------------------------------------------------------
"""

# ==============================================================================
# ЗАВДАННЯ 4. Аналіз активних обмежень (Залишки ресурсів / Slack)
# ==============================================================================

print("=== ЗАВДАННЯ 4. Залишки ресурсів (res.slack) ===")
resource_names = ["Сировина (кг)", "Час обладнання (год)", "Електроенергія (кВт·год)"]
for name, slack in zip(resource_names, res.slack):
    status = "АКТИВНЕ (вузьке місце)" if abs(slack) < 1e-6 else f"НЕАКТИВНЕ (запас: {slack:.2f})"
    print(f"{name}: slack = {slack:.2f} -> {status}")
print()

"""
Інтерпретация активних та неактивних обмежень:
    - Сировина (slack = 0.00): АКТИВНЕ обмеження. Використано повністю 210 кг з 210 кг.
    - Час обладнання (slack = 0.00): АКТИВНЕ обмеження. Використано повністю 250 год з 250 год.
    - Електроенергія (slack = 23.33): НЕАКТИВНЕ обмеження. Залишилося невикористаними 
      23.33 кВт·год із 125 кВт·год.

Бізнес-висновок:
    Сировина та час обладнання є "вузькими місцями" (дефіцитними ресурсами). Для 
    підвищення прибутку підприємству варто інвестувати у закупівлю додаткової сировини 
    або розширення фонду робочого часу обладнання. Інвестиції у збільшення ліміту 
    електроенергії є недоцільними, оскільки цей ресурс і так має надлишок.
--------------------------------------------------------------------------------
"""

# ==============================================================================
# ЗАВДАННЯ 5. What-if аналіз (Зміна ресурсних обмежень)
# ==============================================================================

print("=== ЗАВДАННЯ 5. What-if сценарії ===")

# 1. Збільшення АКТИВНОГО ресурсу (Сировина +15%: 210 * 1.15 = 241.5 кг)
b_ub_active = [241.5, 250, 125]
res_active = linprog(c, A_ub=A_ub, b_ub=b_ub_active, bounds=bounds, method="highs")

print("1) Збільшення АКТИВНОГО ресурсу (Сировина +15% -> 241.5 кг):")
print(f"   Новий res.x: {res_active.x}")
print(f"   Новий прибуток: {-res_active.fun:.2f} грн")
print(f"   Зміна прибутку: +{-res_active.fun - (-res.fun):.2f} грн")

# 2. Збільшення НЕАКТИВНОГО ресурсу (Електроенергія +50%: 125 * 1.50 = 187.5 кВт·год)
b_ub_inactive = [210, 250, 187.5]
res_inactive = linprog(c, A_ub=A_ub, b_ub=b_ub_inactive, bounds=bounds, method="highs")

print("\n2) Збільшення НЕАКТИВНОГО ресурсу (Електроенергія +50% -> 187.5 кВт·год):")
print(f"   Новий res.x: {res_inactive.x}")
print(f"   Новий прибуток: {-res_inactive.fun:.2f} грн")
print(f"   Зміна прибутку: {-res_inactive.fun - (-res.fun):.2f} грн\n")

"""
Пояснення результатів Завдання 5:
    1) Розширення активного ресурсу (сировини) збільшило прибуток з 3472.92 грн до 
       3755.36 грн (на +282.44 грн) і змінило структуру випуску (x1=18.93, x2=67.86). 
       Це підтверджує, що активний ресурс є вузьким місцем, яке обмежує ріст бізнесу.
    2) Розширення неактивного ресурсу (електроенергії) ніяк не змінило ні обсяг випуску 
       (x1=23.75, x2=54.17), ні прибуток (3472.92 грн). Це прямо відповідає теорії ЛП: 
       ресурс із запасом не стримує виробництво, тому його додаткове збільшення не дає 
       економічного ефекту.
--------------------------------------------------------------------------------
"""

# ==============================================================================
# ВІДПОВІДІ НА КОНТРОЛЬНІ ПИТАННЯ
# ==============================================================================

"""
КОНТРОЛЬНІ ПИТАННЯ (ВІДПОВІДІ):

1. Чому scipy.optimize.linprog завжди мінімізує і що робити для максимізації?
   Математичний апарат linprog за замовчуванням реалізує пошук мінімуму функції min c^T*x. 
   Щоб розв'язати задачу максимізації max c^T*x, використовують тотожність: 
   max(c^T*x) = - min(-c^T*x). Для цього коефіцієнти цільової функції c множать на -1, 
   а підсумкове значення res.fun інвертують за знаком (-res.fun).

2. Чому обмеження "не менше" (>=) потрібно домножити на -1?
   Канонічна форма linprog приймає обмеження-нерівності виключно у вигляді "не більше": 
   A_ub * x <= b_ub. Якщо обмеження задане як a*x >= b, то множення обох частин на -1 
   змінює знак нерівності на протилежний: -a*x <= -b.

3. Різниця між активним і неактивним обмеженням:
   - Активне обмеження (slack = 0): ресурс вичерпано повністю в точці оптимуму. 
     Воно є "вузьким місцем", яке фізично утримує прибуток від подальшого зростання.
   - Неактивне обмеження (slack > 0): ресурс вичерпано не повністю, існує невикористаний запас.

4. Узгодженість зміни прибутку в Завданні 5:
   Отримані зміни повністю узгоджуються з теоретичними очікуваннями: збільшення ліміту 
   активного ресурсу (сировини) розширило область допустимих розв'язків і дало приріст 
   прибутку, тоді як розширення неактивного ресурсу (електроенергії) лише збільшило 
   невикористаний за залишок slack, залишивши прибуток без змін.
"""

=== ЗАВДАННЯ 2. Результати linprog ===
res.x (одиниці А та Б): [23.57142857 54.28571429]
res.fun (значення мінус-функції): -3467.857142857143
Дійсний максимальний прибуток (-res.fun): 3467.86 грн
res.status: 0
res.message: Optimization terminated successfully. (HiGHS Status 7: Optimal)

ПІДТВЕРДЖЕННЯ: res.status == 0 (Глобальний оптимум успішно знайдено).

=== ЗАВДАННЯ 4. Залишки ресурсів (res.slack) ===
Сировина (кг): slack = 0.00 -> АКТИВНЕ (вузьке місце)
Час обладнання (год): slack = 0.00 -> АКТИВНЕ (вузьке місце)
Електроенергія (кВт·год): slack = 23.57 -> НЕАКТИВНЕ (запас: 23.57)

=== ЗАВДАННЯ 5. What-if сценарії ===
1) Збільшення АКТИВНОГО ресурсу (Сировина +15% -> 241.5 кг):
   Новий res.x: [19.07142857 67.78571429]
   Новий прибуток: 3760.36 грн
   Зміна прибутку: +292.50 грн

2) Збільшення НЕАКТИВНОГО ресурсу (Електроенергія +50% -> 187.5 кВт·год):
   Новий res.x: [23.57142857 54.28571429]
   Новий прибуток: 3467.86 грн
   Зміна прибутку: 0.00 грн



'\nКОНТРОЛЬНІ ПИТАННЯ (ВІДПОВІДІ):\n\n1. Чому scipy.optimize.linprog завжди мінімізує і що робити для максимізації?\n   Математичний апарат linprog за замовчуванням реалізує пошук мінімуму функції min c^T*x. \n   Щоб розв\'язати задачу максимізації max c^T*x, використовують тотожність: \n   max(c^T*x) = - min(-c^T*x). Для цього коефіцієнти цільової функції c множать на -1, \n   а підсумкове значення res.fun інвертують за знаком (-res.fun).\n\n2. Чому обмеження "не менше" (>=) потрібно домножити на -1?\n   Канонічна форма linprog приймає обмеження-нерівності виключно у вигляді "не більше": \n   A_ub * x <= b_ub. Якщо обмеження задане як a*x >= b, то множення обох частин на -1 \n   змінює знак нерівності на протилежний: -a*x <= -b.\n\n3. Різниця між активним і неактивним обмеженням:\n   - Активне обмеження (slack = 0): ресурс вичерпано повністю в точці оптимуму. \n     Воно є "вузьким місцем", яке фізично утримує прибуток від подальшого зростання.\n   - Неактивне обмеження (slack > 0): р